# Chapter 02 — Exercises: Data Cleaning & Preparing

**Session 1 | Chapter 2 | Core tasks: ~10 minutes · Bonus: if you have time**

You will clean and prepare a small messy **housing dataset** for ML — in the **right order**:

1. Explore & deterministic fixes (spelling, duplicates, impossible values)
2. **Split** train / test
3. Impute missing values — **fit on train**
4. Encode + scale with a `ColumnTransformer` — **fit on train**

Each task has `# TODO` comments. Bonus tasks are at the end.

> This dataset is intentionally different from the demo (student survey). Applying the same techniques to new data is part of the learning.

**When you're done:** compare with `04-solutions/ch02_data_cleaning_solutions.ipynb`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

sns.set_theme(style='whitegrid')
print('Ready!')

## Setup: Load the Dataset

In [ ]:
# Don't modify this cell — just run it!
housing_data = {
    'house_id': list(range(1, 26)),
    'area_sqm': [75, 120, None, 95, 200, 85, 60, 110, None, 140,
                 90, 78, 130, 9999, 105, 88, 72, 115, None, 95,
                 80, 125, 92, 67, 103],
    'rooms': [3, 4, 3, None, 5, 3, 2, 4, 3, 5,
              3, 3, 4, 4, 4, 3, 2, 4, 3, 3,
              3, 4, 3, 2, 4],
    'location': ['urban', 'suburban', 'rural', 'Urban', 'suburban',
                 'URBAN', None, 'rural', 'suburban', 'urban',
                 'Urban', 'rural', 'suburban', 'urban', None,
                 'rural', 'urban', 'suburban', 'rural', 'urban',
                 'suburban', 'urban', 'rural', 'suburban', 'urban'],
    'has_garage': ['yes', 'no', 'YES', 'Yes', 'no', 'yes', None, 'no', 'yes', 'YES',
                   'No', 'yes', 'no', 'yes', 'no', None, 'yes', 'no', 'yes', 'no',
                   'yes', 'YES', 'no', 'yes', 'no'],
    'price_1000_eur': [250, 380, 180, 290, 520, 270, 190, 360, 230, 440,
                       280, 210, 410, 275, 340, 225, 175, 370, 215, 285,
                       240, 395, 268, 165, 315]
}
df = pd.DataFrame(housing_data)
df = pd.concat([df, df.iloc[[3]]], ignore_index=True)   # one accidental duplicate row
print('Dataset loaded! Shape:', df.shape)
df.head()

---
## Task 1: Explore & Deterministic Fixes (~3 min)

Nothing here computes a statistic → safe to do **before** the split.

In [ ]:
# TODO 1a: Print the missing values per column and the number of duplicate rows
# Hint: df.isnull().sum()   and   df.duplicated().sum()

In [ ]:
# TODO 1b: Print the unique values of 'location' and 'has_garage'. What is wrong with them?

In [ ]:
# TODO 1c: Fix the spelling: make 'location' and 'has_garage' lowercase and strip whitespace
# Hint: df['col'] = df['col'].str.lower().str.strip()

In [ ]:
# TODO 1d: Drop duplicate rows (keep the first) and reset the index
# Hint: df = df.drop_duplicates().reset_index(drop=True)

In [ ]:
# TODO 1e: area_sqm = 9999 is a data-entry error (no house has 9999 m²).
#          Set every area_sqm > 1000 to np.nan so it gets imputed later.
# Hint: df.loc[df['area_sqm'] > 1000, 'area_sqm'] = np.nan

In [ ]:
# Check your work — should show 24 rows, clean categories, and NaNs only where expected
print('Shape:', df.shape)
print('location:', df['location'].unique())
print('has_garage:', df['has_garage'].unique())
print(df.isnull().sum())

---
## Task 2: Train / Test Split (~1 min)

From here on everything is a statistic → **split first**.

In [ ]:
feature_cols = ['area_sqm', 'rooms', 'location', 'has_garage']
X = df[feature_cols]
y = df['price_1000_eur']

# TODO 2: Split into train (80%) and test (20%) with random_state=42.
#         Question: should you use stratify=y here? (Hint: what type of target is price?)
# X_train, X_test, y_train, y_test = ...

print('Train size:', X_train.shape[0], '| Test size:', X_test.shape[0])

---
## Task 3: Impute Missing Values — Fit on Train Only (~3 min)

In [ ]:
num_cols = ['area_sqm', 'rooms']
cat_cols = ['location', 'has_garage']

# TODO 3a: Create a SimpleImputer(strategy='median') for the numeric columns.
#          fit_transform on X_train[num_cols], transform on X_test[num_cols]
# num_imputer = ...
# X_train[num_cols] = ...
# X_test[num_cols]  = ...

# TODO 3b: Same for the categorical columns with strategy='most_frequent'
# cat_imputer = ...


# Check: which values did the imputer learn from the TRAIN set?
print('numeric fill values:', dict(zip(num_cols, num_imputer.statistics_)))
print('categorical fill values:', dict(zip(cat_cols, cat_imputer.statistics_)))
print('Missing left — train:', X_train.isnull().sum().sum(), '| test:', X_test.isnull().sum().sum())

---
## Task 4: Encode + Scale With a `ColumnTransformer` (~3 min)

Fill in the two blanks: which transformer goes on the numeric columns, which on the categorical ones?

In [ ]:
# TODO 4a: complete the ColumnTransformer
#   numeric columns     → StandardScaler()
#   categorical columns → OneHotEncoder(handle_unknown='ignore')
preprocess = ColumnTransformer([
    ('num', ..., num_cols),
    ('cat', ..., cat_cols),
])

# TODO 4b: fit_transform on X_train, transform on X_test  (never fit on test!)
# X_train_ready = ...
# X_test_ready  = ...

print('Output columns:', list(preprocess.get_feature_names_out()))
print('X_train_ready shape:', X_train_ready.shape)
print('X_test_ready shape: ', X_test_ready.shape)
print('\n✅ Data is clean, encoded, scaled — and the test set never leaked into any statistic!')

---
## Bonus (if you have time)

Pick any — helper code is provided.

In [ ]:
# BONUS A: Outliers with the IQR rule — computed on TRAIN, applied to both.
#   Compute Q1, Q3, IQR of X_train['area_sqm'], the bounds Q1-1.5*IQR / Q3+1.5*IQR,
#   and clip X_train['area_sqm'] and X_test['area_sqm'] to these bounds. How many train rows were affected?

In [ ]:
# BONUS B: Compare imputation strategies. Re-run Task 3 with strategy='mean' instead of 'median'.
#   Which fill value changes the most? Why is median more robust? (Hint: what if 9999 had NOT been removed?)

In [ ]:
# BONUS C: Z-score outliers. z = (x - mean) / std computed on TRAIN.
#   Which train rows have |z| > 3 for area_sqm? Compare with the IQR result from Bonus A.
# z = (X_train['area_sqm'] - X_train['area_sqm'].mean()) / X_train['area_sqm'].std()

In [ ]:
# BONUS D: Correlation heatmap of the cleaned numeric TRAIN features + price.
#   Which feature correlates most with price?
# corr = pd.concat([X_train[num_cols], y_train], axis=1).corr()
# sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1)

---
**Solutions:** `../04-solutions/ch02_data_cleaning_solutions.ipynb`